In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, TimestampType, DoubleType, FloatType

catalog_name="ecommerce"

df=spark.read.table(f"{catalog_name}.silver.slv_order_items")

display(df.limit(4))

add new required columns- gross, discount, sales columns

In [0]:
#add gross amount

df=df.withColumn("gross_amount",F.col("quantity") * F.col("unit_price"))

#add discount amount

df=df.withColumn("discount_amount",
                 F.ceil(F.col("gross_amount") * (F.col("discount_pct") / 100.0))
)

#add sales amount

df=df.withColumn("sales_amount",F.col("gross_amount") - F.col("discount_amount") + F.col("tax_amount"))

#add date id

df=df.withColumn("date_id",F.date_format(F.col("dt"),"yyyyMMdd").cast(IntegerType())) #create date key

#coupon flag
# coupon flag= 1 if coupon code is not null else 0
df=df.withColumn("coupon_flag",
                 F.when(F.col("coupon_code").isNotNull(),F.lit(1))
                 .otherwise(F.lit(0))
)

df.limit(4).display()

currency conversion

In [0]:
# define your fixed rates as of 2026-04-21
fx_rates={
    "INR":1.00,
    "AED":25.46,
    "AUD":66.96,
    "CAD":68.47,
    "GBP":126.44,
    "SGD":73.46,
    "USD":93.62,
}

rates=[(k,float(v)) for k,v in fx_rates.items()]
rates_df=spark.createDataFrame(rates,["currency","inr_rate"])
rates_df.show()

In [0]:
df=(
    df.
    join(
        rates_df,
        rates_df.currency == F.upper(F.trim(F.col("unit_price_currency"))),
        "left"
    )
    .withColumn("sales_amount_inr",F.col("sales_amount") * F.col("inr_rate"))
    .withColumn("sales_amount_inr",F.ceil(F.col("sales_amount_inr")))
)

df.limit(4).display()

In [0]:
orders_gold_df=df.select(
    F.col("date_id"),
    F.col("dt").alias("transaction_date"),
    F.col("order_ts").alias("transaction_ts"),
    F.col("order_id").alias("transaction_id"),
    F.col("customer_id"),
    F.col("item_seq").alias("seq_no"),
    F.col("product_id"),
    F.col(  "channel"),
    F.col("coupon_code"),
    F.col("coupon_flag"),
    F.col("unit_price_currency"),
    F.col("quantity"),
    F.col("unit_price"),
    F.col("gross_amount"),
    F.col("discount_pct").alias("discount_percent"),
    F.col("discount_amount"),
    F.col("tax_amount"),
    F.col("sales_amount").alias("net_amount"),
    F.col("sales_amount_inr").alias("net_amount_inr"),
    F.col("currency"),
)

In [0]:
orders_gold_df.write.format("delta").mode("overwrite").option("mergeSchema","true").\
    saveAsTable(f"{catalog_name}.gold.gld_fact_order_items")

In [0]:
#sanity check

spark.sql(f"SELECT count(*) FROM {catalog_name}.gold.gld_fact_order_items").show()